In [1]:
from sklearn.model_selection import KFold
import numpy as np
import pandas as pd

from factor_analyzer import FactorAnalyzer
from sklearn.mixture import GaussianMixture

print("="*60)
print("VALIDACIÓN CRUZADA DE RESULTADOS")
print("="*60)

VALIDACIÓN CRUZADA DE RESULTADOS


In [2]:
# 0. CARGAR DATASET
# =======================================
# df = pd.read_csv("../data_collection_peru_csv/data_merge/COLLECT_COMP_PERU.csv")
df = pd.read_csv("../data_ready/COLLECT_COMP_dataset_treated.csv")
df_afe = pd.read_csv("../Aprendisaje/df_afe.csv")
df_acl_scaled = pd.read_csv("../Aprendisaje/df_acl_scaled.csv")
# df_with_factors = pd.read_csv('../Aprendisaje/df_with_factors.csv')

n_factors = 5
n_classes_final = 7

In [3]:
# Configurar validación cruzada
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Almacenar resultados
cv_results = {
    'afe_stability': [],
    'acl_agreement': [],
    'irt_correlation': []
}

for fold, (train_idx, test_idx) in enumerate(kf.split(df_afe)):
    print(f"\nFold {fold+1}/5...")
    
    # 1. Validar AFE
    train_data = df_afe.iloc[train_idx]
    test_data = df_afe.iloc[test_idx]
    
    # AFE en train
    fa_train = FactorAnalyzer(n_factors=n_factors, rotation='varimax')
    fa_train.fit(train_data)
    
    # AFE en test
    fa_test = FactorAnalyzer(n_factors=n_factors, rotation='varimax')
    fa_test.fit(test_data)
    
    # Correlación de cargas
    loadings_corr = np.corrcoef(fa_train.loadings_.flatten(), 
                                fa_test.loadings_.flatten())[0,1]
    cv_results['afe_stability'].append(loadings_corr)
    
    # 2. Validar ACL
    gmm_train = GaussianMixture(n_components=n_classes_final, random_state=42)
    gmm_train.fit(df_acl_scaled.iloc[train_idx])
    
    # Predecir en test
    train_labels = gmm_train.predict(df_acl_scaled.iloc[train_idx])
    test_labels = gmm_train.predict(df_acl_scaled.iloc[test_idx])
    
    # Estabilidad de clasificación (usando silhouette)
    from sklearn.metrics import silhouette_score
    sil_score = silhouette_score(df_acl_scaled.iloc[test_idx], test_labels)
    cv_results['acl_agreement'].append(sil_score)
    
    print(f"  AFE stability: {loadings_corr:.3f}")
    print(f"  ACL silhouette: {sil_score:.3f}")

# Resumen
print("\n" + "="*60)
print("RESUMEN DE VALIDACIÓN CRUZADA")
print("="*60)

for metric, values in cv_results.items():
    if values:  # Si hay valores
        mean_val = np.mean(values)
        std_val = np.std(values)
        print(f"{metric}:")
        print(f"  Media: {mean_val:.3f}")
        print(f"  Desv. Est.: {std_val:.3f}")
        
        # Interpretación
        if metric == 'afe_stability' and mean_val > 0.8:
            print("  ✓ Estructura factorial estable")
        elif metric == 'acl_agreement' and mean_val > 0.3:
            print("  ✓ Clases latentes consistentes")


Fold 1/5...


c:\Users\max_a\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\max_a\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


  AFE stability: 0.993
  ACL silhouette: 0.364

Fold 2/5...


c:\Users\max_a\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\max_a\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


  AFE stability: 0.990
  ACL silhouette: 0.364

Fold 3/5...


c:\Users\max_a\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\max_a\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


  AFE stability: 0.697
  ACL silhouette: 0.366

Fold 4/5...


c:\Users\max_a\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\max_a\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


  AFE stability: 0.699
  ACL silhouette: 0.340

Fold 5/5...


c:\Users\max_a\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\max_a\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


  AFE stability: 0.638
  ACL silhouette: 0.358

RESUMEN DE VALIDACIÓN CRUZADA
afe_stability:
  Media: 0.803
  Desv. Est.: 0.155
  ✓ Estructura factorial estable
acl_agreement:
  Media: 0.358
  Desv. Est.: 0.010
  ✓ Clases latentes consistentes
